# Financial Risk Analysis with Python  
## Task 4: Financial Risk Identification  

**Objective:**  
To identify customer accounts exhibiting potential financial risk by analyzing overdrafts, large withdrawals, balance volatility, and anomalous transaction behavior using statistical techniques.


In [2]:
# Load cleaned transaction data for Task 4
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_transactions.csv")

# Ensure transaction date is datetime
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])

df.head()


,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,TransactionAmount,AccountBalance,RiskScore,CreditRating,TenureMonths,year,month
0,78,CUST1223,ACC33287,credit,withdrawal,Savings Account,Firm E,South,Manager 4,2023-06-01,81300.425190,40843.56193,0.330474,484,13,2023.0,2023-06
1,21,CUST8266,ACC58667,savings,withdrawal,Credit Card,Firm A,South,Manager 2,NaT,9269.640373,61183.03953,0.089688,836,18,NaN,NaN
2,176,CUST9420,ACC99117,credit,transfer,Home Loan,Firm A,East,Manager 1,NaT,28138.552650,85460.13405,0.340010,451,25,NaN,NaN
3,167,CUST5253,ACC10117,loan,payment,Mutual Fund,Firm D,Central,Manager 1,NaT,83943.556980,100525.35900,0.605383,487,13,NaN,NaN
4,46,CUST1223,ACC74631,savings,deposit,Credit Card,Firm A,East,Manager 4,2023-12-05,77104.456470,57425.69930,1.042441,393,10,2023.0,2023-12


In [3]:
# Identify frequent large withdrawals and overdraft incidents

# Standardize transaction type
df['TransactionType'] = df['TransactionType'].str.lower().str.strip()

# Define large withdrawal threshold (95th percentile)
large_withdrawal_threshold = df.loc[
    df['TransactionType'] == 'withdrawal', 'TransactionAmount'
].quantile(0.95)

# Flag large withdrawals
df['large_withdrawal_flag'] = (
    (df['TransactionType'] == 'withdrawal') &
    (df['TransactionAmount'] >= large_withdrawal_threshold)
)

# Flag overdraft incidents (negative balance)
df['overdraft_flag'] = df['AccountBalance'] < 0

# Count risk events per account
risk_events = (
    df.groupby('AccountID')[['large_withdrawal_flag', 'overdraft_flag']]
      .sum()
      .reset_index()
)

risk_events.head()


,AccountID,large_withdrawal_flag,overdraft_flag
0,ACC10117,0,0
1,ACC10996,0,0
2,ACC11062,0,0
3,ACC11188,0,0
4,ACC11285,0,0


In [4]:
# Calculate balance volatility per account

# Balance volatility using standard deviation
balance_volatility = (
    df.groupby('AccountID')['AccountBalance']
      .std()
      .reset_index(name='balance_std_dev')
)

# Optional: Coefficient of Variation (CV) for scale-free volatility
avg_balance = (
    df.groupby('AccountID')['AccountBalance']
      .mean()
      .reset_index(name='avg_balance')
)

balance_volatility = balance_volatility.merge(avg_balance, on='AccountID')
balance_volatility['balance_cv'] = (
    balance_volatility['balance_std_dev'] / balance_volatility['avg_balance'].abs()
)

balance_volatility.head()


,AccountID,balance_std_dev,avg_balance,balance_cv
0,ACC10117,38036.539989,94082.590727,0.404289
1,ACC10996,30036.385595,72517.761136,0.414194
2,ACC11062,33831.704345,73541.328302,0.460037
3,ACC11188,9067.866694,51359.039950,0.176558
4,ACC11285,25590.622179,79923.619870,0.320188


In [5]:
# Detect anomalous transaction amounts using IQR method

# Calculate IQR for transaction amounts
Q1 = df['TransactionAmount'].quantile(0.25)
Q3 = df['TransactionAmount'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds for anomalies
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Flag anomalous transactions
df['anomaly_flag'] = (
    (df['TransactionAmount'] < lower_bound) |
    (df['TransactionAmount'] > upper_bound)
)

# Identify accounts with anomalous behavior
anomalous_accounts = (
    df[df['anomaly_flag']]
    [['AccountID']]
    .drop_duplicates()
)

anomalous_accounts.head()


,AccountID
9,ACC19178
161,ACC35419
370,ACC77533
525,ACC92104
533,ACC23736


In [6]:
# Highlight customers with irregular or suspicious transaction behavior

# Combine multiple risk indicators
suspicious_customers = df[
    (df['anomaly_flag'] == True) |
    (df['large_withdrawal_flag'] == True) |
    (df['overdraft_flag'] == True)
][['AccountID']].drop_duplicates()

suspicious_customers.head()


,AccountID
9,ACC19178
47,ACC21264
55,ACC50817
60,ACC24070
115,ACC34431
